# Chapter 15 - Text Preprocessing and Feature Extraction

Machine learning algorithms operate on numbers, not words. The fundamental challenge of
Natural Language Processing (NLP) is turning **raw text into numerical feature vectors**
that preserve the information a model needs to make predictions. This notebook covers the
essential preprocessing pipeline and the two most widely used feature extraction methods:
Bag of Words and TF-IDF.

**Topics covered:**
- Why text needs preprocessing before ML
- The text preprocessing pipeline: lowercase, punctuation removal, tokenization, stop words, stemming/lemmatization
- Bag of Words model with `CountVectorizer`
- Visualizing word frequencies
- TF-IDF: why term frequency alone is not enough
- `TfidfVectorizer` in scikit-learn
- Comparing BoW vs TF-IDF representations
- N-grams: capturing word sequences with `ngram_range`
- Sparse matrices: why text features are stored as sparse
- Vocabulary control: `max_features`, `min_df`, `max_df`
- Practical example: preprocessing a collection of documents

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

np.random.seed(42)
%matplotlib inline

---
## 1. The Core Problem: Text is Not Numbers

A sentence like *"The movie was absolutely brilliant!"* is meaningful to us, but to a
machine learning algorithm it is just a string of characters. We need a systematic way to
represent text as fixed-length numerical vectors so we can feed it into classifiers,
regressors, or clustering algorithms.

The typical NLP workflow looks like this:

```
Raw text  -->  Preprocessing  -->  Feature extraction  -->  Numerical matrix  -->  ML model
```

Preprocessing cleans and normalizes the text. Feature extraction converts the cleaned
text into numbers. This notebook focuses on both steps.

---
## 2. A Synthetic Document Collection

We will work with a small, hand-crafted collection of documents throughout this notebook.
Using synthetic data means we can see exactly what is happening at every step without
needing external downloads.

In [ ]:
documents = [
    "The cat sat on the mat and the cat was happy.",
    "Dogs are loyal animals. Dogs love to play fetch!",
    "The weather is sunny today. A perfect day for a walk.",
    "Machine learning algorithms learn from data.",
    "Python is a great programming language for data science.",
    "Cats and dogs are popular pets around the world.",
    "Natural language processing turns text into numbers.",
    "The quick brown fox jumps over the lazy dog.",
    "Data science involves statistics, programming, and domain knowledge.",
    "I love programming in Python. It is simple and powerful.",
]

for i, doc in enumerate(documents):
    print(f"Doc {i}: {doc}")

---
## 3. The Text Preprocessing Pipeline

Before extracting features, we clean the raw text. A standard pipeline includes:

1. **Lowercasing** — "The" and "the" should be treated as the same word.
2. **Punctuation removal** — periods, commas, and exclamation marks carry little information for most tasks.
3. **Tokenization** — splitting a string into individual words (tokens).
4. **Stop word removal** — discarding extremely common words ("the", "is", "and") that appear in nearly every document.
5. **Stemming or Lemmatization** — reducing words to their root form ("running" -> "run").

Let us implement each step from scratch to understand what is happening, then see how
scikit-learn handles it automatically.

### 3.1 Lowercasing

In [ ]:
sample = "The Cat SAT on the Mat!"
print(f"Original:    {sample}")
print(f"Lowercased:  {sample.lower()}")

### 3.2 Punctuation Removal

In [ ]:
def remove_punctuation(text):
    """Remove all punctuation characters from text."""
    return text.translate(str.maketrans('', '', string.punctuation))

sample_lower = sample.lower()
sample_clean = remove_punctuation(sample_lower)
print(f"After punctuation removal: {sample_clean}")

### 3.3 Tokenization

In [ ]:
tokens = sample_clean.split()
print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

### 3.4 Stop Word Removal

Stop words are words so common that they appear in almost every document and therefore
carry very little discriminative information. Scikit-learn ships with a built-in English
stop word list.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

print(f"Number of sklearn stop words: {len(ENGLISH_STOP_WORDS)}")
print(f"Sample stop words: {sorted(list(ENGLISH_STOP_WORDS))[:20]}")

In [ ]:
filtered_tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
print(f"Original tokens:  {tokens}")
print(f"After stop words: {filtered_tokens}")
print(f"Removed: {set(tokens) - set(filtered_tokens)}")

### 3.5 Stemming and Lemmatization

Both reduce words to a common root form. The difference:

- **Stemming** chops off suffixes using crude rules (fast, but can produce non-words like "happi").
- **Lemmatization** uses vocabulary and morphological analysis to return a proper dictionary word ("better" -> "good").

We will demonstrate a simple suffix-stripping stemmer. In production you might use
NLTK's PorterStemmer or spaCy's lemmatizer, but for this notebook we keep dependencies
minimal.

In [ ]:
def simple_stem(word):
    """A naive suffix-stripping stemmer for demonstration."""
    suffixes = ['ing', 'tion', 'ly', 'ed', 'er', 'est', 'ness', 's']
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word

test_words = ['running', 'dogs', 'happily', 'played', 'processing', 'cats', 'greater']
for w in test_words:
    print(f"  {w:15s} -> {simple_stem(w)}")

### 3.6 Full Pipeline: Putting It All Together

In [ ]:
def preprocess(text):
    """Apply the full preprocessing pipeline to a string."""
    text = text.lower()
    text = remove_punctuation(text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    tokens = [simple_stem(t) for t in tokens]
    return tokens

for i, doc in enumerate(documents[:3]):
    print(f"Doc {i} original:    {doc}")
    print(f"Doc {i} preprocessed: {preprocess(doc)}")
    print()

---
## 4. Bag of Words: CountVectorizer

The **Bag of Words** (BoW) model is the simplest way to convert text to numbers:

1. Build a **vocabulary** — the set of all unique words across all documents.
2. For each document, count how many times each vocabulary word appears.
3. Represent each document as a vector of these counts.

The result is a **document-term matrix** where rows are documents and columns are words.

It is called "bag of words" because **word order is completely ignored** — the document
is treated as an unordered collection (bag) of words.

In [ ]:
# Basic CountVectorizer — it handles lowercasing and tokenization internally
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(documents)

print(f"Document-term matrix shape: {bow_matrix.shape}")
print(f"  {bow_matrix.shape[0]} documents")
print(f"  {bow_matrix.shape[1]} unique words (vocabulary size)")
print(f"\nVocabulary (first 20): {list(count_vec.vocabulary_.keys())[:20]}")

In [ ]:
# View the full document-term matrix as a DataFrame
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=[f'Doc {i}' for i in range(len(documents))]
)
bow_df

In [ ]:
# Notice: "the" appears many times — it dominates several documents
print("Word counts for 'the' across documents:")
print(bow_df['the'].values)
print(f"\nTotal occurrences of 'the': {bow_df['the'].sum()}")
print(f"Total occurrences of 'data': {bow_df['data'].sum()}")

---
## 5. Visualizing Word Frequencies

Looking at the most frequent words in a corpus is a useful sanity check. Very common
words (like "the") often dominate — which is exactly why we use stop word removal
or TF-IDF weighting.

In [ ]:
# Total word counts across all documents
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 15 words
word_counts.head(15).plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title('Top 15 Most Frequent Words')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Distribution of word frequencies
axes[1].hist(word_counts.values, bins=range(1, int(word_counts.max()) + 2),
             color='steelblue', edgecolor='k', align='left')
axes[1].set_title('Distribution of Word Frequencies')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Number of words')

plt.tight_layout()
plt.show()

Most words appear only once or twice — this is typical of natural language and is known
as **Zipf's law**: a few words are extremely common, and the vast majority are rare.

---
## 6. CountVectorizer with Stop Words

Scikit-learn's `CountVectorizer` can handle stop word removal for us with a single
parameter.

In [ ]:
count_vec_stop = CountVectorizer(stop_words='english')
bow_stop = count_vec_stop.fit_transform(documents)

print(f"Without stop words: {bow_matrix.shape[1]} features")
print(f"With stop words:    {bow_stop.shape[1]} features")
print(f"Removed {bow_matrix.shape[1] - bow_stop.shape[1]} stop words from vocabulary")

In [ ]:
bow_stop_df = pd.DataFrame(
    bow_stop.toarray(),
    columns=count_vec_stop.get_feature_names_out(),
    index=[f'Doc {i}' for i in range(len(documents))]
)
bow_stop_df

Now "the", "is", "and", "on", etc. are gone, and the remaining words are more informative.

---
## 7. TF-IDF: Why Term Frequency Alone Is Not Enough

Bag of Words counts how often a word appears in a document (**term frequency**, TF). But
a word that appears in *every* document is not useful for distinguishing between documents.
The word "the" might appear 5 times in a document, but it tells us nothing about the
topic.

**TF-IDF** (Term Frequency - Inverse Document Frequency) solves this by multiplying the
term frequency by a weight that **penalizes words that appear in many documents**:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

where:
- $\text{TF}(t, d)$ = number of times term $t$ appears in document $d$
- $\text{IDF}(t) = \log\frac{1 + n}{1 + \text{df}(t)} + 1$ (scikit-learn's smoothed version)
- $n$ = total number of documents
- $\text{df}(t)$ = number of documents containing term $t$

**Intuition:** A word that appears frequently in one document but rarely across the corpus
gets a high TF-IDF score — it is likely a discriminative, topic-specific word.

In [ ]:
# Manual TF-IDF calculation for one word to build intuition
n_docs = len(documents)

# Word: "cat" appears in documents 0 and 5
tf_cat_doc0 = 2   # "cat" appears twice in Doc 0
df_cat = 2         # "cat" appears in 2 documents
idf_cat = np.log((1 + n_docs) / (1 + df_cat)) + 1
tfidf_cat_doc0 = tf_cat_doc0 * idf_cat

# Word: "the" appears in documents 0, 2, 5, 7
tf_the_doc0 = 3   # "the" appears three times in Doc 0
df_the = 4         # "the" appears in 4 documents
idf_the = np.log((1 + n_docs) / (1 + df_the)) + 1
tfidf_the_doc0 = tf_the_doc0 * idf_the

print(f"'cat':  TF={tf_cat_doc0}, DF={df_cat}, IDF={idf_cat:.4f}, TF-IDF={tfidf_cat_doc0:.4f}")
print(f"'the':  TF={tf_the_doc0}, DF={df_the}, IDF={idf_the:.4f}, TF-IDF={tfidf_the_doc0:.4f}")
print()
print("Even though 'the' has higher TF, 'cat' gets a higher IDF because it is rarer.")

---
## 8. TfidfVectorizer in Scikit-Learn

`TfidfVectorizer` combines tokenization, counting, and TF-IDF weighting in a single step.
By default it also applies L2 normalization to each row so that document length does not
bias the representation.

In [ ]:
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(documents)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Type: {type(tfidf_matrix)}")

In [ ]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f'Doc {i}' for i in range(len(documents))]
)
tfidf_df.round(3)

In [ ]:
# Compare: "the" vs "algorithms" in TF-IDF
print("TF-IDF scores for 'the':")
print(tfidf_df['the'].round(4).values)
print(f"\nTF-IDF scores for 'algorithms':")
print(tfidf_df['algorithms'].round(4).values)
print()
print("'algorithms' only appears in Doc 3, so it gets a high TF-IDF there.")
print("'the' is spread across many documents, so its TF-IDF is lower per document.")

---
## 9. Comparing Bag of Words vs TF-IDF

Let us put the two representations side by side for a single document.

In [ ]:
doc_idx = 0  # "The cat sat on the mat and the cat was happy."

# Get non-zero features for this document
bow_vals = bow_df.iloc[doc_idx]
tfidf_vals = tfidf_df.iloc[doc_idx]

# Only show words that appear in this document
mask = bow_vals > 0

comparison = pd.DataFrame({
    'BoW (raw count)': bow_vals[mask],
    'TF-IDF (weighted)': tfidf_vals[mask].round(4)
}).sort_values('BoW (raw count)', ascending=False)

print(f"Document: \"{documents[doc_idx]}\"\n")
print(comparison.to_string())

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

comparison['BoW (raw count)'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_title(f'Bag of Words — Doc {doc_idx}')
axes[0].set_ylabel('Raw count')
axes[0].tick_params(axis='x', rotation=45)

comparison['TF-IDF (weighted)'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='k')
axes[1].set_title(f'TF-IDF — Doc {doc_idx}')
axes[1].set_ylabel('TF-IDF score')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('BoW vs TF-IDF for the same document', fontsize=13)
plt.tight_layout()
plt.show()

In Bag of Words, "the" dominates because it appears 3 times. In TF-IDF, "cat", "mat",
"sat", and "happy" get boosted because they are rarer across the corpus — exactly the
words that tell us what this document is *about*.

---
## 10. N-Grams: Capturing Word Sequences

The Bag of Words model treats each word independently, which loses word order. Consider:

- "not good" — negative sentiment
- "good" — positive sentiment

A unigram (single-word) model cannot tell these apart. **N-grams** are contiguous
sequences of *n* words:

- **Unigram** (n=1): "not", "good"
- **Bigram** (n=2): "not good"
- **Trigram** (n=3): "is not good"

Scikit-learn's vectorizers support n-grams via the `ngram_range` parameter.

In [ ]:
# Unigrams only (default)
vec_uni = CountVectorizer(ngram_range=(1, 1))
vec_uni.fit(documents)
print(f"Unigrams only:          {len(vec_uni.vocabulary_)} features")

# Unigrams + bigrams
vec_bi = CountVectorizer(ngram_range=(1, 2))
vec_bi.fit(documents)
print(f"Unigrams + bigrams:     {len(vec_bi.vocabulary_)} features")

# Unigrams + bigrams + trigrams
vec_tri = CountVectorizer(ngram_range=(1, 3))
vec_tri.fit(documents)
print(f"Uni + bi + trigrams:    {len(vec_tri.vocabulary_)} features")

In [ ]:
# Show some bigram features
bigram_features = [f for f in vec_bi.get_feature_names_out() if ' ' in f]
print(f"Number of bigram features: {len(bigram_features)}")
print(f"\nSample bigrams: {bigram_features[:15]}")

In [ ]:
# N-grams with TF-IDF
tfidf_bi = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_bi_matrix = tfidf_bi.fit_transform(documents)

print(f"TF-IDF with bigrams shape: {tfidf_bi_matrix.shape}")

# Top bigram features by total TF-IDF weight
total_weight = np.array(tfidf_bi_matrix.sum(axis=0)).flatten()
feature_names = tfidf_bi.get_feature_names_out()
bigram_mask = [' ' in f for f in feature_names]

bigram_weights = pd.Series(
    total_weight[bigram_mask],
    index=feature_names[bigram_mask]
).sort_values(ascending=False)

print("\nTop 10 bigrams by total TF-IDF weight:")
print(bigram_weights.head(10).to_string())

**Trade-off:** Adding higher-order n-grams captures more context but dramatically
increases the vocabulary size (and therefore the dimensionality of the feature space).
In practice, `ngram_range=(1, 2)` is a popular default that balances expressiveness
and efficiency.

---
## 11. Sparse Matrices: Why Text Features Are Stored Sparse

A typical document uses only a tiny fraction of the total vocabulary. In our example,
each document has roughly 8-10 words out of a vocabulary of ~50. In real-world corpora
with vocabularies of 50,000+ words, over 99% of entries are zero.

Storing all those zeros as a regular NumPy array would be extremely wasteful. Instead,
scikit-learn's vectorizers return **scipy sparse matrices** that only store the non-zero
entries.

In [ ]:
import scipy.sparse as sp

print(f"Type: {type(tfidf_matrix)}")
print(f"Shape: {tfidf_matrix.shape}")
print(f"Total elements: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]}")
print(f"Non-zero elements: {tfidf_matrix.nnz}")
print(f"Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%}")

In [ ]:
# Memory comparison
dense_memory = tfidf_matrix.toarray().nbytes
sparse_memory = (tfidf_matrix.data.nbytes + tfidf_matrix.indices.nbytes + 
                 tfidf_matrix.indptr.nbytes)

print(f"Dense array memory:  {dense_memory:,} bytes")
print(f"Sparse matrix memory: {sparse_memory:,} bytes")
print(f"Ratio: {dense_memory / sparse_memory:.1f}x")
print()
print("For our small dataset the savings are modest.")
print("For a real corpus with 50k vocabulary and 10k documents,")
print(f"a dense matrix would need ~{50000 * 10000 * 8 / 1e9:.1f} GB!")

In [ ]:
# Visualize the sparsity pattern
fig, ax = plt.subplots(figsize=(14, 4))
ax.spy(tfidf_matrix, markersize=5, aspect='auto', color='steelblue')
ax.set_xlabel('Feature (word) index')
ax.set_ylabel('Document index')
ax.set_title('Sparsity pattern of the TF-IDF matrix')
plt.tight_layout()
plt.show()

Each dot is a non-zero entry. Most of the matrix is empty — this is the norm for text
data and is exactly why sparse storage is essential at scale.

---
## 12. Vocabulary Control: max_features, min_df, max_df

In real corpora the vocabulary can be enormous. Scikit-learn provides three knobs to
control it:

| Parameter | What it does | Typical use |
|-----------|-------------|-------------|
| `max_features` | Keep only the top N most frequent terms | Hard cap on dimensionality |
| `min_df` | Ignore terms that appear in fewer than N documents (int) or fraction of documents (float) | Remove very rare words (typos, proper nouns) |
| `max_df` | Ignore terms that appear in more than N documents (int) or fraction of documents (float) | Remove corpus-wide common words (alternative to stop words) |

In [ ]:
# max_features: keep only the top 10 words
vec_max = CountVectorizer(max_features=10)
vec_max.fit(documents)
print(f"max_features=10: {vec_max.get_feature_names_out()}")

In [ ]:
# min_df: ignore words that appear in fewer than 2 documents
vec_mindf = CountVectorizer(min_df=2)
vec_mindf.fit(documents)
print(f"min_df=2: {len(vec_mindf.vocabulary_)} features")
print(f"Kept words: {vec_mindf.get_feature_names_out()}")

In [ ]:
# max_df: ignore words that appear in more than 50% of documents
vec_maxdf = CountVectorizer(max_df=0.5)
vec_maxdf.fit(documents)
print(f"max_df=0.5: {len(vec_maxdf.vocabulary_)} features")
print(f"Kept words: {list(vec_maxdf.get_feature_names_out()[:20])}...")

In [ ]:
# Combining parameters
vec_combined = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20,
    min_df=1,
    max_df=0.8
)
X_combined = vec_combined.fit_transform(documents)

print(f"Combined parameters: {X_combined.shape[1]} features")
print(f"Features: {list(vec_combined.get_feature_names_out())}")

---
## 13. Practical Example: Preprocessing a Document Collection

Let us bring everything together on a slightly larger synthetic corpus. We will create
documents about three topics — sports, technology, and cooking — and see how BoW and
TF-IDF represent them.

In [ ]:
corpus = [
    # Sports
    "The team scored three goals in the second half of the match.",
    "The coach praised the players for their excellent performance on the field.",
    "The championship final will be played at the national stadium next week.",
    "The striker scored a brilliant goal from outside the penalty area.",
    "Training sessions focus on fitness, passing, and tactical awareness.",
    # Technology
    "The new smartphone features a faster processor and improved camera.",
    "Artificial intelligence is transforming how we interact with software.",
    "Cloud computing allows businesses to scale their infrastructure on demand.",
    "The latest update includes bug fixes and performance improvements.",
    "Developers are building applications using machine learning frameworks.",
    # Cooking
    "Preheat the oven to 180 degrees before placing the dish inside.",
    "Season the chicken with salt, pepper, and a squeeze of lemon juice.",
    "Stir the sauce gently until it thickens, then remove from heat.",
    "Fresh ingredients make a significant difference in the final flavor.",
    "The recipe calls for two cups of flour and one cup of sugar.",
]

topics = ['sports'] * 5 + ['technology'] * 5 + ['cooking'] * 5

for topic, doc in zip(topics, corpus):
    print(f"[{topic:10s}] {doc}")

In [ ]:
# TF-IDF with sensible defaults
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=50)
X_corpus = tfidf.fit_transform(corpus)

print(f"Corpus shape: {X_corpus.shape}")
print(f"Features: {list(tfidf.get_feature_names_out())}")

In [ ]:
# Show the top TF-IDF words for each topic group
feature_names = tfidf.get_feature_names_out()
X_dense = X_corpus.toarray()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, topic, start in zip(axes, ['Sports', 'Technology', 'Cooking'], [0, 5, 10]):
    # Average TF-IDF across documents in this topic
    mean_tfidf = X_dense[start:start+5].mean(axis=0)
    top_idx = mean_tfidf.argsort()[-10:][::-1]
    
    ax.barh(range(10), mean_tfidf[top_idx], color='steelblue', edgecolor='k')
    ax.set_yticks(range(10))
    ax.set_yticklabels(feature_names[top_idx])
    ax.invert_yaxis()
    ax.set_title(f'Top words: {topic}')
    ax.set_xlabel('Mean TF-IDF')

plt.suptitle('Most important words per topic (TF-IDF)', fontsize=13)
plt.tight_layout()
plt.show()

TF-IDF naturally surfaces the topic-specific words: "scored", "goal", "match" for sports;
"software", "computing", "learning" for technology; "sauce", "oven", "recipe" for cooking.
Generic words like "the" and "is" have been removed by the stop word filter and would have
been down-weighted by IDF anyway.

In [ ]:
# Visualize document similarity using TF-IDF vectors
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(X_corpus)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(15))
ax.set_yticks(range(15))
ax.set_xticklabels([f'{t[0].upper()}{i}' for i, t in enumerate(topics)], rotation=45)
ax.set_yticklabels([f'{t[0].upper()}{i}' for i, t in enumerate(topics)])
ax.set_title('Document Similarity (Cosine) based on TF-IDF')
plt.colorbar(im, ax=ax)

# Draw topic group borders
for pos in [4.5, 9.5]:
    ax.axhline(y=pos, color='black', linewidth=2)
    ax.axvline(x=pos, color='black', linewidth=2)

plt.tight_layout()
plt.show()

The block-diagonal pattern confirms that documents within the same topic are more
similar to each other than to documents from different topics — exactly what we want
before feeding these features into a classifier.

---
## 14. Summary

| Concept | Key takeaway |
|---------|-------------|
| **Preprocessing** | Lowercase, remove punctuation, tokenize, remove stop words, stem/lemmatize — reduces noise and vocabulary size |
| **Bag of Words** | Represents documents as word-count vectors; simple but ignores word order and over-weights common words |
| **TF-IDF** | Weights word counts by inverse document frequency, highlighting discriminative terms |
| **N-grams** | Captures short word sequences to partially restore word order information |
| **Sparse matrices** | Text features are extremely sparse; scipy sparse matrices save memory |
| **Vocabulary control** | `max_features`, `min_df`, `max_df` let you tune the feature space size |

**Next up:** [02 - Text Classification](02_text_classification.ipynb) — using these
features to build classifiers with Naive Bayes, Logistic Regression, and SVM.